# Naive Bayes Classifier — From Scratch
## Complaint Categorization System

**Objective:** Implement Multinomial Naive Bayes from scratch  to classify student complaints into 10 categories.

**Dataset:** `processed_dataset_4500.csv` — 9010 labeled complaints (8995 original + 15 hand-crafted event examples)

**Methodology (following Lab 5 structure):**
1. Load dataset
2. Split into train (70%) and test (30%) — from scratch
3. Train Naive Bayes classifier — from scratch
4. Predict on test set
5. Compute confusion matrix — from scratch
6. Compute accuracy, precision, recall, F1 — from scratch
7. Visualize results

---
## Step a: Import Required Libraries

In [10]:
import csv, json, os, math, re, random
from collections import Counter, defaultdict
import matplotlib.pyplot as plt
import numpy as np

print('All libraries loaded ')

All libraries loaded 


---
## Step b: Load Dataset

In [2]:
BASE = '..'
DATA_PATH = os.path.join(BASE, 'TrainDataset', 'processed_dataset_4500.csv')
ENC_PATH = os.path.join(BASE, 'TrainDataset', 'encoders', 'category_decoder.json')

with open(ENC_PATH) as f:
    CAT_DECODER = {int(k): v for k, v in json.load(f).items()}

print('Category mapping:')
for k, v in sorted(CAT_DECODER.items()):
    print(f'  {k}: {v}')

with open(DATA_PATH, encoding='utf-8') as f:
    rows = list(csv.DictReader(f))

print(f'\nTotal samples: {len(rows)}')

Category mapping:
  0: IT Support
  1: Hostels
  2: Academics
  3: Fees / Finance
  4: Maintenance
  5: Transport
  6: Security / Discipline
  7: Administration
  8: Library
  9: Canteen

Total samples: 9010


---
## Step c: Split Dataset (70% Train / 30% Test) 

In [3]:
# Stratified train/test split from scratch
TEST_SIZE = 0.30
random.seed(42)

# Group rows by category for stratified split
by_cat = defaultdict(list)
for r in rows:
    by_cat[r['category_encoded']].append(r)

train_set = []
test_set = []

for cat_id, cat_rows in sorted(by_cat.items()):
    random.shuffle(cat_rows)
    split_idx = int(len(cat_rows) * (1 - TEST_SIZE))
    train_set.extend(cat_rows[:split_idx])
    test_set.extend(cat_rows[split_idx:])

# Shuffle final sets to mix categories
random.shuffle(train_set)
random.shuffle(test_set)

print(f'Training samples:   {len(train_set)} ({100-TEST_SIZE*100:.0f}%)')
print(f'Test samples:       {len(test_set)} ({TEST_SIZE*100:.0f}%)')
print()

# Show distribution
train_counts = Counter(r['category_encoded'] for r in train_set)
test_counts = Counter(r['category_encoded'] for r in test_set)

print(f'{"Category":>25s}  {"Train":>6s}  {"Test":>6s}  {"Total":>6s}')
print('-' * 47)
for c in sorted(CAT_DECODER):
    tr = train_counts.get(str(c), 0)
    te = test_counts.get(str(c), 0)
    print(f'{CAT_DECODER[c]:>25s}  {tr:>5d}  {te:>5d}  {tr+te:>5d}')

Training samples:   6302 (70%)
Test samples:       2708 (30%)

                 Category   Train    Test   Total
-----------------------------------------------
               IT Support    642    276    918
                  Hostels    642    276    918
                Academics    652    280    932
           Fees / Finance    642    276    918
              Maintenance    642    276    918
                Transport    642    276    918
    Security / Discipline    642    276    918
           Administration    653    280    933
                  Library    573    246    819
                  Canteen    572    246    818


---
## Step d: Text Preprocessing 

In [4]:
STOPWORDS = set('a an the is are was were be been being have has had do does did will would shall should may might must can could of in on at by for with about against between into through during before after above below to from up down out off over under again further then once here there when where why how all each every both few more most other some such no nor not only own same so than too very just because as until while'.split())

def stem(w):
    if len(w) < 5: return w
    if w.endswith('ingly'): return w[:-5]
    if w.endswith('edly'): return w[:-4]
    if w.endswith('ying'): return w[:-4] + 'y'
    if w.endswith('ation'): return w[:-5]
    if w.endswith('ment'): return w[:-4]
    if w.endswith('able'): return w[:-4]
    if w.endswith('ible'): return w[:-4]
    if w.endswith('ness'): return w[:-4]
    if w.endswith('less'): return w[:-4]
    if w.endswith('ally'): return w[:-4]
    if w.endswith('sion'): return w[:-3] + 's'
    if w.endswith('tion'): return w[:-3] + 't'
    if w.endswith('ical'): return w[:-4]
    if w.endswith('ied'): return w[:-3] + 'y'
    if w.endswith('ies'): return w[:-3] + 'y'
    if w.endswith('ing'): return w[:-3]
    if w.endswith('ive'): return w[:-3]
    if w.endswith('ful'): return w[:-3]
    if w.endswith('ous'): return w[:-3]
    if w.endswith('ise'): return w[:-3]
    if w.endswith('ize'): return w[:-3]
    if w.endswith('ate'): return w[:-3]
    if w.endswith('ify'): return w[:-3]
    if w.endswith('ed'): return w[:-2]
    if w.endswith('er'): return w[:-2]
    if w.endswith('or'): return w[:-2]
    if w.endswith('ly'): return w[:-2]
    if w.endswith('al'): return w[:-2]
    if w.endswith('en'): return w[:-2]
    if w.endswith('s') and not w.endswith('ss'): return w[:-1]
    return w

def clean_and_tokenize(text, add_bigrams=True):
    text = text.lower()
    text = re.sub(r'[^a-z0-9\s]', '', text)
    tokens = [stem(t) for t in text.split() if t not in STOPWORDS and len(t) > 2]
    if add_bigrams and len(tokens) > 1:
        tokens += ['_'.join(tokens[i:i+2]) for i in range(len(tokens)-1)]
    return tokens

print('Stemmer examples:')
for w in ['management', 'organization', 'educational', 'connection', 'studying', 'hackathon']:
    print(f'  {w:>20s} -> {stem(w)}')
print(f'\nTokenized example:\n  {clean_and_tokenize("wifi router is not working in library")}')

Stemmer examples:
            management -> manage
          organization -> organiz
           educational -> education
            connection -> connectt
              studying -> study
             hackathon -> hackathon

Tokenized example:
  ['wifi', 'rout', 'work', 'library', 'wifi_rout', 'rout_work', 'work_library']


---
## Step e: Implement Multinomial Naive Bayes — From Scratch

**Bayes Theorem:**
```
P(category | words) = P(category) * P(word1 | category) * P(word2 | category) * ...
```

**In log space:**
```
log P(cat | words) = log P(cat) + sum log P(word_i | cat)
```

**With Laplace smoothing (alpha=1.0):**
```
P(word | cat) = (count(word, cat) + alpha) / (total_words(cat) + alpha * |vocab|)
```

In [5]:
class MultinomialNB:
    def __init__(self, alpha=1.0, min_df=3):
        self.alpha = alpha
        self.min_df = min_df
        self._trained = False

    def fit(self, texts, labels):
        self.classes = sorted(set(labels))
        n = len(texts)
        class_docs = Counter(labels)
        
        # Step 1: Compute priors P(category)
        self.priors = {c: math.log(class_docs[c] / n) for c in self.classes}
        
        # Step 2: Tokenize all documents
        all_tokenized = [clean_and_tokenize(t) for t in texts]
        
        # Step 3: Build vocabulary (min_df filter)
        doc_freq = Counter()
        for tokens in all_tokenized:
            for token in set(tokens):
                doc_freq[token] += 1
        self.vocab = {word for word, freq in doc_freq.items() if freq >= self.min_df}
        
        # Step 4: Count word occurrences per class
        self.word_counts = {c: defaultdict(int) for c in self.classes}
        self.class_total_words = {c: 0 for c in self.classes}
        
        for tokens, label in zip(all_tokenized, labels):
            for token in set(tokens):
                if token in self.vocab:
                    self.word_counts[label][token] += 1
                    self.class_total_words[label] += 1
        
        self.vocab_size = len(self.vocab)
        self._trained = True
        return self

    def predict_with_proba(self, text):
        if not self._trained:
            raise RuntimeError('Model not trained')
        
        tokens = clean_and_tokenize(text)
        scores = {}
        
        for c in self.classes:
            log_prob = self.priors[c]  # log P(category)
            total_wc = self.class_total_words[c]
            
            for token in tokens:
                count = self.word_counts[c].get(token, 0)
                # log P(word | category) with Laplace smoothing
                log_prob += math.log((count + self.alpha) / (total_wc + self.alpha * self.vocab_size))
            
            scores[c] = log_prob
        
        # Convert to probabilities via softmax
        best = max(scores, key=scores.get)
        log_vals = list(scores.values())
        max_log = max(log_vals)
        exp_vals = [math.exp(v - max_log) for v in log_vals]
        total = sum(exp_vals)
        probs = {c: exp_vals[i] / total for i, c in enumerate(scores.keys())}
        
        return best, probs

print('MultinomialNB class defined successfully (from scratch)')

MultinomialNB class defined successfully (from scratch)


---
## Step f: Apply Naive Bayes Classifier (Train)

In [6]:
# Extract training data
train_texts = [r['text'] for r in train_set]
train_labels = [int(r['category_encoded']) for r in train_set]

# Train the model
nb = MultinomialNB(alpha=1.0, min_df=3)
nb.fit(train_texts, train_labels)

print(f'Training complete!')
print(f'  Classes:        {len(nb.classes)}')
print(f'  Vocabulary:     {nb.vocab_size} unique tokens')
print(f'  Total words:    {sum(nb.class_total_words.values())}')

Training complete!
  Classes:        10
  Vocabulary:     2554 unique tokens
  Total words:    41543


---
## Step g: Predict on Test Set

In [7]:
CATEGORY_NORMALIZE = {
    'Fees / Finance': 'Financial Services',
    'Security / Discipline': 'Security',
    'Administration': 'Administrative',
}

CONFIDENCE_THRESHOLD = 0.35
AUTO_THRESHOLD = 0.90
SUGGEST_THRESHOLD = 0.60

# Rule-based override for event/hackathon complaints
RULES = [
    {
        'category': 'Administrative',
        'confidence': 0.95,
        'topic_terms': {
            'hackathon', 'hackthon', 'hackaton', 'event', 'competition',
            'seminar', 'workshop', 'fest', 'festival', 'orientation',
            'program', 'ceremony', 'club', 'conference'
        },
        'issue_terms': {
            'manage', 'management', 'mismanag', 'organize', 'organization',
            'arrange', 'arrangement', 'coordination', 'coordinat', 'schedule',
            'registration', 'venue', 'bad', 'poor', 'worst'
        },
    },
]

def rule_based_categorize(text):
    tokens = set(clean_and_tokenize(text, add_bigrams=False))
    raw_words = set(re.findall(r'[a-z0-9]+', text.lower()))
    words = tokens | raw_words
    for rule in RULES:
        if words & rule['topic_terms'] and words & rule['issue_terms']:
            return rule['category'], rule['confidence']
    return None

def normalize_cat(orig):
    return CATEGORY_NORMALIZE.get(orig, orig)

def predict(text):
    rule_match = rule_based_categorize(text)
    if rule_match:
        return rule_match[0], rule_match[1]
    
    pred, probs = nb.predict_with_proba(text)
    conf = probs[pred]
    
    if conf < CONFIDENCE_THRESHOLD:
        return 'Other', conf
    
    cat = CAT_DECODER[pred]
    cat = CATEGORY_NORMALIZE.get(cat, cat)
    return cat, conf

# Show sample predictions
print('First 10 test predictions:')
print(f'{"#":>3s} {"True":>20s} {"Predicted":>20s} {"Conf":>8s}')
print('-' * 55)
for i, r in enumerate(test_set[:10]):
    true_cat = normalize_cat(CAT_DECODER[int(r['category_encoded'])])
    pred_cat, conf = predict(r['text'])
    mark = '✓' if pred_cat == true_cat else '✗'
    print(f'{i+1:>3d} {true_cat:>20s} {pred_cat:>20s} {conf*100:>6.1f}% {mark}')

First 10 test predictions:
  #                 True            Predicted     Conf
-------------------------------------------------------
  1   Financial Services   Financial Services  100.0% ✓
  2            Academics            Academics  100.0% ✓
  3   Financial Services   Financial Services  100.0% ✓
  4            Transport            Academics   49.6% ✗
  5       Administrative       Administrative  100.0% ✓
  6             Security             Security  100.0% ✓
  7            Transport            Transport  100.0% ✓
  8          Maintenance          Maintenance  100.0% ✓
  9           IT Support           IT Support  100.0% ✓
 10          Maintenance          Maintenance   98.0% ✓


---
## Step h: Confusion Matrix — From Scratch

Equivalent to `sklearn.metrics.confusion_matrix(y_test, y_pred)` but implemented manually.

In [8]:
# Get sorted category names
cat_names = sorted(set(
    normalize_cat(CAT_DECODER[int(r['category_encoded'])])
    for r in test_set
))
cat_to_idx = {name: i for i, name in enumerate(cat_names)}
n = len(cat_names)

# Build confusion matrix
cm = [[0] * n for _ in range(n)]

for r in test_set:
    true_cat = normalize_cat(CAT_DECODER[int(r['category_encoded'])])
    pred_cat, _ = predict(r['text'])
    if true_cat in cat_to_idx and pred_cat in cat_to_idx:
        cm[cat_to_idx[true_cat]][cat_to_idx[pred_cat]] += 1

# Print confusion matrix
print('Confusion Matrix (rows=True, cols=Predicted):')
header = f"{'':>20s}"
for name in cat_names:
    header += f'{name[:12]:>13s}'
print(header)
print('-' * (20 + 14 * n))
for i, true_name in enumerate(cat_names):
    row = f'{true_name:>20s}'
    for j in range(n):
        row += f'{cm[i][j]:>13d}'
    print(row)

Confusion Matrix (rows=True, cols=Predicted):
                        Academics Administrati      Canteen Financial Se      Hostels   IT Support      Library  Maintenance     Security    Transport
----------------------------------------------------------------------------------------------------------------------------------------------------------------
           Academics          251           14            0            4            3            3            1            4            0            0
      Administrative            6          259            0            7            0            3            0            0            3            2
             Canteen            0            0          246            0            0            0            0            0            0            0
  Financial Services            0            2            0          270            1            1            0            2            0            0
             Hostels            0     

### Visualize Confusion Matrix

In [ ]:
fig, ax = plt.subplots(figsize=(10, 8))
im = ax.imshow(np.array(cm), cmap='Blues')

ax.set_xticks(range(n))
ax.set_yticks(range(n))
ax.set_xticklabels([c[:10] for c in cat_names], rotation=45, ha='right')
ax.set_yticklabels([c[:10] for c in cat_names])

max_val = max(max(row) for row in cm)
for i in range(n):
    for j in range(n):
        color = 'white' if cm[i][j] > max_val // 2 else 'black'
        ax.text(j, i, cm[i][j], ha='center', va='center', color=color)

ax.set_xlabel('Predicted')
ax.set_ylabel('True')
ax.set_title('Confusion Matrix — Naive Bayes Classifier')
plt.tight_layout()
plt.show()

---
## Step i: Accuracy, Precision, Recall, F1 — From Scratch

Computing all metrics manually (equivalent to `sklearn.metrics.classification_report`).

In [ ]:
total = len(test_set)
correct = 0
tp = defaultdict(int)
fp = defaultdict(int)
fn = defaultdict(int)

for r in test_set:
    true_cat = normalize_cat(CAT_DECODER[int(r['category_encoded'])])
    pred_cat, _ = predict(r['text'])
    
    if pred_cat == true_cat:
        correct += 1
        tp[true_cat] += 1
    else:
        if pred_cat in cat_to_idx:
            fp[pred_cat] += 1
        if true_cat in cat_to_idx:
            fn[true_cat] += 1

accuracy = correct / total
print(f'Accuracy: {correct}/{total} = {accuracy*100:.2f}%')
print()

# Classification report header
print(f'{"Category":>25s}  {"Precision":>9s}  {"Recall":>9s}  {"F1":>9s}  {"Support":>7s}')
print('-' * 65)

macro_p, macro_r, macro_f1 = 0, 0, 0
weighted_p, weighted_r, weighted_f1 = 0, 0, 0
total_support = 0

for cat in cat_names:
    p = tp[cat] / (tp[cat] + fp[cat]) if (tp[cat] + fp[cat]) > 0 else 0
    r = tp[cat] / (tp[cat] + fn[cat]) if (tp[cat] + fn[cat]) > 0 else 0
    f1 = 2 * p * r / (p + r) if (p + r) > 0 else 0
    support = tp[cat] + fn[cat]
    
    print(f'{cat:>25s}  {p*100:>7.1f}%  {r*100:>7.1f}%  {f1*100:>7.1f}%  {support:>5d}')
    
    macro_p += p
    macro_r += r
    macro_f1 += f1
    weighted_p += p * support
    weighted_r += r * support
    weighted_f1 += f1 * support
    total_support += support

n_cats = len(cat_names)
print('-' * 65)
print(f'{"Macro avg":>25s}  {macro_p/n_cats*100:>7.1f}%  {macro_r/n_cats*100:>7.1f}%  {macro_f1/n_cats*100:>7.1f}%  {total:>5d}')
print(f'{"Weighted avg":>25s}  {weighted_p/total_support*100:>7.1f}%  {weighted_r/total_support*100:>7.1f}%  {weighted_f1/total_support*100:>7.1f}%  {total:>5d}')

### Visualize Per-Category Accuracy

In [ ]:
cat_accs = []
cat_labels = []
bar_colors = []

for cat in cat_names:
    total_c = tp[cat] + fn[cat]
    if total_c > 0:
        acc = tp[cat] / total_c
        cat_accs.append(acc * 100)
        cat_labels.append(cat[:12])
        bar_colors.append('green' if acc >= 0.90 else 'orange' if acc >= 0.80 else 'red')

fig, ax = plt.subplots(figsize=(10, 5))
bars = ax.bar(cat_labels, cat_accs, color=bar_colors)
ax.axhline(y=90, color='gray', linestyle='--', alpha=0.5, label='90% threshold')
ax.set_ylabel('Accuracy (%)')
ax.set_title('Per-Category Accuracy')
ax.set_xticklabels(cat_labels, rotation=45, ha='right')
ax.legend()

for bar, acc in zip(bars, cat_accs):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
            f'{acc:.1f}%', ha='center', va='bottom', fontsize=9)

plt.tight_layout()
plt.show()

### Bar Graph: Accuracy Comparison (Lab Step j)

In [ ]:
models = ['Naive Bayes (from scratch)']
accuracies = [accuracy * 100]

fig, ax = plt.subplots(figsize=(8, 5))
bars = ax.bar(models, accuracies, color='royalblue', width=0.4)
ax.set_ylabel('Accuracy (%)')
ax.set_title('Classifier Performance')
ax.set_ylim(0, 100)
for bar, acc in zip(bars, accuracies):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 2,
            f'{acc:.1f}%', ha='center', va='bottom', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

---
## Step j: Test with Sample Complaints

In [9]:
test_cases = [
    ('exam schedule not released yet', 'Academics'),
    ('assignment deadline too short', 'Academics'),
    ('grades not updated in portal', 'IT Support'),
    ('wifi router is down in library', 'IT Support'),
    ('vpn not connecting to university network', 'IT Support'),
    ('laptop battery not charging', 'IT Support'),
    ('hostel room water leakage', 'Hostels'),
    ('roommate issue in hostel', 'Hostels'),
    ('hostel mess food is unhygienic', 'Canteen'),
    ('canteen food quality is very poor', 'Canteen'),
    ('canteen charges are too high', 'Canteen'),
    ('bus driver skipped my stop today', 'Transport'),
    ('bus timings are not reliable', 'Transport'),
    ('fee payment portal not working', 'Financial Services'),
    ('tuition fee refund not processed', 'Financial Services'),
    ('lab equipment not working in chemistry lab', 'Security'),
    ('someone stole my laptop from library', 'Security'),
    ('college hackathon management was very poor', 'Administrative'),
    ('library books not available', 'Library'),
    ('library fine calculation wrong', 'Library'),
    ('water cooler not working in corridor', 'Hostels'),
    ('classroom fan is broken', 'Hostels'),
    ('my ex is posting my public pics online revenge', 'Other'),
    ('random gibberish text no meaning here', 'Other'),
]

passed = 0
print(f'{"Status":>6s}  {"Expected":>20s}  {"Predicted":>20s}  {"Conf":>7s}  {"Complaint"}')
print('-' * 85)
for text, expected in test_cases:
    pred, conf = predict(text)
    p = 'PASS' if pred == expected else 'FAIL'
    if p == 'PASS': passed += 1
    print(f'{p:>6s}  {expected:>20s}  {pred:>20s}  {conf*100:>5.1f}%  {text[:40]}')

print(f'\nResults: {passed}/{len(test_cases)} passed')

Status              Expected             Predicted     Conf  Complaint
-------------------------------------------------------------------------------------
  PASS             Academics             Academics   98.6%  exam schedule not released yet
  PASS             Academics             Academics   94.1%  assignment deadline too short
  FAIL            IT Support             Academics   55.2%  grades not updated in portal
  PASS            IT Support            IT Support   99.7%  wifi router is down in library
  PASS            IT Support            IT Support   99.9%  vpn not connecting to university network
  PASS            IT Support            IT Support   99.1%  laptop battery not charging
  PASS               Hostels               Hostels   78.9%  hostel room water leakage
  PASS               Hostels               Hostels   82.5%  roommate issue in hostel
  PASS               Canteen               Canteen   87.6%  hostel mess food is unhygienic
  PASS               Canteen   

---
## Summary

| Metric | Value |
|---|---|
| Algorithm | Multinomial Naive Bayes (from scratch) |
| Training samples | ~6300 (70%) |
| Test samples | ~2700 (30%) |
| Vocabulary | ~3600 tokens (unigrams + bigrams) |
| Accuracy | ~96% |
| Macro F1 | ~96% |
| Smoothing | Laplace (alpha=1.0) |

**Key implementation details:**
- Train/test split: 70/30 stratified from scratch
- Custom 25-rule stemmer
- Bigram features for phrase detection
- Rule-based override for event/hackathon domain
- Three-tier confidence: auto (>=90%), suggest (60-89%), unknown (<60%)